In [1]:
"""
mlb_scraper_ner.py  —  MLB Trade Rumors Scraper + NER Pipeline
================================================================
Run this module INDEPENDENTLY (in your own environment with internet
access) to produce:

    outputs/team_interest_cache.csv

That CSV is then consumed by the main v3 modelling pipeline. This
separation means the slow, network-dependent scraping step only needs
to run once and can be re-used across model iterations.

Dependencies (install in your environment):
    pip install requests beautifulsoup4 spacy
    python -m spacy download en_core_web_lg

Usage:
    python mlb_scraper_ner.py --years 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025
    python mlb_scraper_ner.py --years 2016 2025   # shorthand: start and end year
    python mlb_scraper_ner.py --from-cache         # skip scraping, re-run NER on saved HTML
"""

import argparse
import json
import os
import re
import time
import logging
from pathlib import Path
from typing import Optional

import pandas as pd

# ── Conditional imports: fail gracefully if not installed ────────────────────
try:
    import requests
    from bs4 import BeautifulSoup
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print("[WARNING] requests/beautifulsoup4 not installed — scraping disabled.")

try:
    import spacy
    HAS_SPACY = True
except ImportError:
    HAS_SPACY = False
    print("[WARNING] spacy not installed — NLP layer will use rule-based only.")


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL    = "https://www.mlbtraderumors.com"
CACHE_DIR   = Path("mlbtr_html_cache")      # raw HTML stored here per page
OUTPUT_DIR  = Path("outputs")
OUTPUT_CSV  = OUTPUT_DIR / "team_interest_cache.csv"

# Be polite: wait this many seconds between HTTP requests.
# MLBTR is a small site; hammering it with requests would be inconsiderate
# and would get your IP throttled.
REQUEST_DELAY_SECS = 1.5

# Hot stove months: Oct–Feb spanning the offseason.
# E.g. contract_year=2022 covers Oct 2022 – Feb 2023.
HOT_STOVE_MONTHS = [(0, 10), (0, 11), (0, 12), (1, 1), (1, 2)]
# Tuples are (year_offset, month) — year_offset 0 = contract year, 1 = following year.

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# TEAM ALIAS TABLE
# ─────────────────────────────────────────────────────────────────────────────
#
# Maps every reasonable way a beat writer might name a team to its
# canonical 3-letter abbreviation. Ordered so longer/more-specific
# strings are checked first to avoid partial matches (e.g. "New York"
# matching before "New York Yankees").

TEAM_ALIASES: dict[str, str] = {
    # AL East
    "New York Yankees": "NYY", "Yankees": "NYY", "Bronx Bombers": "NYY",
    "Boston Red Sox": "BOS", "Red Sox": "BOS",
    "Tampa Bay Rays": "TBR", "Rays": "TBR",
    "Toronto Blue Jays": "TOR", "Blue Jays": "TOR",
    "Baltimore Orioles": "BAL", "Orioles": "BAL",
    # AL Central
    "Chicago White Sox": "CWS", "White Sox": "CWS",
    "Cleveland Guardians": "CLE", "Cleveland Indians": "CLE", "Guardians": "CLE", "Indians": "CLE",
    "Detroit Tigers": "DET", "Tigers": "DET",
    "Kansas City Royals": "KCR", "Royals": "KCR",
    "Minnesota Twins": "MIN", "Twins": "MIN",
    # AL West
    "Houston Astros": "HOU", "Astros": "HOU",
    "Los Angeles Angels": "LAA", "Angels": "LAA",
    "Oakland Athletics": "OAK", "Athletics": "OAK", "A's": "OAK",
    "Seattle Mariners": "SEA", "Mariners": "SEA",
    "Texas Rangers": "TEX", "Rangers": "TEX",
    # NL East
    "Atlanta Braves": "ATL", "Braves": "ATL",
    "Miami Marlins": "MIA", "Marlins": "MIA",
    "New York Mets": "NYM", "Mets": "NYM",
    "Philadelphia Phillies": "PHI", "Phillies": "PHI",
    "Washington Nationals": "WSN", "Nationals": "WSN", "Nats": "WSN",
    # NL Central
    "Chicago Cubs": "CHC", "Cubs": "CHC",
    "Cincinnati Reds": "CIN", "Reds": "CIN",
    "Milwaukee Brewers": "MIL", "Brewers": "MIL",
    "Pittsburgh Pirates": "PIT", "Pirates": "PIT",
    "St. Louis Cardinals": "STL", "Cardinals": "STL",
    # NL West
    "Arizona Diamondbacks": "ARI", "Diamondbacks": "ARI", "D-backs": "ARI",
    "Colorado Rockies": "COL", "Rockies": "COL",
    "Los Angeles Dodgers": "LAD", "Dodgers": "LAD",
    "San Diego Padres": "SDP", "Padres": "SDP",
    "San Francisco Giants": "SFG", "Giants": "SFG",
}

# Sort by descending key length so "New York Yankees" is matched before "Yankees"
_SORTED_ALIASES = sorted(TEAM_ALIASES.items(), key=lambda kv: -len(kv[0]))

# Large-market teams that receive disproportionate rumor coverage;
# used as a binary feature to down-weight their frequency inflation.
BIG_MARKET_TEAMS = {"NYY", "LAD", "BOS", "CHC", "NYM", "SFG", "PHI"}

# Verb patterns that indicate genuine pursuit (not casual mention)
PURSUIT_PATTERN = re.compile(
    r'interest(?:ed)?\s+in'
    r'|pursu(?:ing|es|ed)'
    r'|linked?\s+to'
    r'|meet(?:ing)?\s+with'
    r'|offer(?:ed|ing)?'
    r'|target(?:ing|ed)?'
    r'|negotiat(?:ing|ed|ions?)'
    r'|sign(?:ing|ed)?\s+(?:with|free\s+agent)'
    r'|consider(?:ing|ed)?'
    r'|reach(?:ed|ing)?\s+out'
    r'|scout(?:ing|ed)?',
    re.IGNORECASE,
)



In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# SCRAPER
# ─────────────────────────────────────────────────────────────────────────────

def _fetch(url: str) -> str:
    """
    Fetch a URL with a polite delay and a descriptive User-Agent.
    Caches raw HTML to disk so we never re-fetch the same page.
    Returns the HTML string.
    """
    assert HAS_REQUESTS, "requests not installed"

    # Derive a safe filename from the URL for caching
    safe_name = re.sub(r'[^\w]', '_', url.replace(BASE_URL, '').strip('/')) or 'index'
    cache_file = CACHE_DIR / f"{safe_name}.html"

    if cache_file.exists():
        log.debug(f"Cache hit: {cache_file}")
        return cache_file.read_text(encoding="utf-8", errors="replace")

    log.info(f"Fetching {url}")
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (academic research project; "
            "contact: your@email.com)"  # ← replace with your email so MLBTR
                                         #   can reach you if they have concerns
        )
    }
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file.write_text(resp.text, encoding="utf-8")

    time.sleep(REQUEST_DELAY_SECS)  # polite delay after every real request
    return resp.text


def scrape_month_index(year: int, month: int) -> list[dict]:
    """
    Scrape the MLBTR monthly archive page and return a list of
    {url, date, title, text} dicts — one per article.
    """
    url  = f"{BASE_URL}/{year}/{month:02d}/"
    html = _fetch(url)
    soup = BeautifulSoup(html, "html.parser")

    posts = []
    for article in soup.select("article"):
        title_el = article.select_one("h2.entry-title a")
        date_el  = article.select_one("time")
        body_el  = article.select_one("div.entry-content")

        if not (title_el and body_el):
            continue

        posts.append({
            "url":   title_el["href"],
            "date":  date_el["datetime"] if date_el else None,
            "title": title_el.get_text(strip=True),
            "text":  body_el.get_text(separator=" ", strip=True),
        })

    log.info(f"  {year}-{month:02d}: {len(posts)} posts")
    return posts


def scrape_post_tags(post_url: str) -> list[str]:
    """
    Each MLBTR post has human-curated tags for players and teams in
    the post footer. These tags are more reliable than free-text NER
    because a human editor applied them — use them as a ground-truth
    cross-check against NER results.
    """
    html = _fetch(post_url)
    soup = BeautifulSoup(html, "html.parser")
    return [a.get_text(strip=True) for a in soup.select("a[rel='tag']")]


def collect_hot_stove_posts(contract_years: list[int]) -> pd.DataFrame:
    """
    For each contract year, scrape Oct–Feb of that offseason.

    Returns a DataFrame with columns:
        contract_year, url, date, title, text, tags (list)
    """
    all_posts = []
    for cy in contract_years:
        for (yr_offset, mo) in HOT_STOVE_MONTHS:
            yr = cy + yr_offset
            try:
                posts = scrape_month_index(yr, mo)
                for p in posts:
                    p["contract_year"] = cy
                all_posts.extend(posts)
            except Exception as e:
                log.warning(f"  Failed {yr}-{mo:02d}: {e}")

    df = pd.DataFrame(all_posts)
    log.info(f"Total posts scraped: {len(df):,}")
    return df



In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# RULE-BASED TEAM EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def _sentences(text: str) -> list[str]:
    """Split text into sentences using a simple regex boundary."""
    return re.split(r'(?<=[.!?])\s+', text)


def extract_teams_rulebased(text: str, player_last_name: str) -> set[str]:
    """
    Scan sentences that:
      1. Contain the player's last name
      2. Contain a pursuit-signal verb

    Then extract any team alias found in that sentence.

    Returns a set of canonical team abbreviations.
    """
    found = set()
    for sent in _sentences(text):
        # Only process sentences that mention the player
        if player_last_name.lower() not in sent.lower():
            continue
        # Only process sentences with genuine pursuit language
        if not PURSUIT_PATTERN.search(sent):
            continue
        # Extract team aliases — sorted longest-first to avoid partial matches
        for alias, abbrev in _SORTED_ALIASES:
            if re.search(r'\b' + re.escape(alias) + r'\b', sent, re.IGNORECASE):
                found.add(abbrev)
    return found


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# spaCy NER LAYER (second pass — catches what rule-based misses)
# ─────────────────────────────────────────────────────────────────────────────

_nlp = None   # lazy-loaded so import doesn't fail if spaCy is absent

def _get_nlp():
    """Load the spaCy model once and cache it."""
    global _nlp
    if _nlp is None:
        assert HAS_SPACY, "spacy not installed"
        _nlp = spacy.load("en_core_web_lg")
    return _nlp


def extract_teams_spacy(text: str, player_last_name: str) -> set[str]:
    """
    Use spaCy's named-entity recognition to find ORG entities in
    sentences that mention the player. Then map recognised ORG strings
    back to canonical team abbreviations via the alias table.

    This catches paraphrases like "the AL East club" less reliably, but
    catches clean team names that the rule-based system might miss due
    to punctuation or hyphenation.
    """
    if not HAS_SPACY:
        return set()

    nlp  = _get_nlp()
    doc  = nlp(text)
    found = set()

    for sent in doc.sents:
        if player_last_name.lower() not in sent.text.lower():
            continue
        if not PURSUIT_PATTERN.search(sent.text):
            continue
        for ent in sent.ents:
            if ent.label_ == "ORG":
                for alias, abbrev in _SORTED_ALIASES:
                    if alias.lower() in ent.text.lower():
                        found.add(abbrev)
    return found


def extract_teams_combined(text: str, player_last_name: str) -> set[str]:
    """
    Union of rule-based and spaCy results.
    Rule-based is always run; spaCy only when available.
    """
    teams = extract_teams_rulebased(text, player_last_name)
    if HAS_SPACY:
        teams |= extract_teams_spacy(text, player_last_name)
    return teams



In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# FINE-TUNING ANNOTATION HELPER
# ─────────────────────────────────────────────────────────────────────────────
#
# To improve NER accuracy you can fine-tune spaCy on labeled MLBTR sentences.
# This function bootstraps training data from the rule-based system, which you
# then review and correct before feeding to spaCy's training loop.
#
# Recommended annotation tool: Label Studio (free, web-based)
#   https://labelstud.io/

def bootstrap_training_data(
    posts_df: pd.DataFrame,
    n_samples: int = 500,
    output_path: str = "ner_bootstrap_labels.json",
) -> list[dict]:
    """
    Sample sentences from scraped posts, auto-label them with the
    rule-based extractor, and save to JSON in spaCy's training format.

    You then open the JSON, manually correct mistakes, and run:
        python -m spacy train config.cfg --paths.train ner_bootstrap_labels.json

    Parameters
    ----------
    posts_df   : DataFrame from collect_hot_stove_posts()
    n_samples  : how many sentences to include
    output_path: where to write the bootstrap labels

    Returns
    -------
    list of spaCy-format training dicts
    """
    training_data = []
    sample = posts_df.sample(min(n_samples * 5, len(posts_df)), random_state=42)

    for _, row in sample.iterrows():
        for sent in _sentences(row.get("text", "")):
            if not PURSUIT_PATTERN.search(sent):
                continue

            entities = []
            for alias, abbrev in _SORTED_ALIASES:
                for m in re.finditer(r'\b' + re.escape(alias) + r'\b', sent, re.IGNORECASE):
                    entities.append((m.start(), m.end(), "MLB_TEAM"))

            if entities:
                training_data.append({
                    "text": sent,
                    "entities": sorted(entities),   # spaCy requires sorted, non-overlapping
                })

            if len(training_data) >= n_samples:
                break
        if len(training_data) >= n_samples:
            break

    with open(output_path, "w") as f:
        json.dump(training_data, f, indent=2)

    log.info(f"Bootstrap labels written to {output_path} ({len(training_data)} examples)")
    return training_data


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# AGGREGATE TO PER-PLAYER FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def build_team_interest_features(
    posts_df: pd.DataFrame,
    player_name: str,
    contract_year: int,
) -> dict:
    """
    Aggregate all hot-stove posts for a given player + contract year into
    a flat feature dict.

    Features produced
    -----------------
    n_teams_interested     : count of unique teams linked to the player
    n_rumor_mentions       : total post count mentioning the player
    has_big_market_interest: 1 if any of NYY/LAD/BOS/CHC/NYM/SFG/PHI appears
    first_rumor_doy        : day-of-year of the first mention (early = hotter market)
    rumor_intensity        : n_teams_interested × n_rumor_mentions (interaction)
    """
    last_name = player_name.split()[-1]

    # Filter to posts that mention the player in this contract year's offseason
    relevant = posts_df[
        (posts_df["contract_year"] == contract_year) &
        (posts_df["text"].str.contains(last_name, case=False, na=False))
    ]

    if relevant.empty:
        return {
            "n_teams_interested":      0,
            "n_rumor_mentions":        0,
            "has_big_market_interest": 0,
            "first_rumor_doy":         365,  # no rumours = late / silent
            "rumor_intensity":         0,
        }

    # Extract teams across all relevant posts
    all_teams: set[str] = set()
    for _, row in relevant.iterrows():
        all_teams |= extract_teams_combined(row["text"], last_name)

    # Parse dates for timing feature
    dates = pd.to_datetime(relevant["date"], errors="coerce").dropna()
    first_doy = int(dates.dt.dayofyear.min()) if len(dates) > 0 else 365

    n_teams   = len(all_teams)
    n_mentions = len(relevant)

    return {
        "n_teams_interested":      n_teams,
        "n_rumor_mentions":        n_mentions,
        "has_big_market_interest": int(bool(all_teams & BIG_MARKET_TEAMS)),
        "first_rumor_doy":         first_doy,
        "rumor_intensity":         n_teams * n_mentions,
    }


def build_all_team_interest(
    contracts_df: pd.DataFrame,
    posts_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Run build_team_interest_features for every row in contracts_df.
    Returns a DataFrame indexed by (mlbam_id, contract_year) with the
    5 team-interest features.
    """
    rows = []
    total = len(contracts_df)
    for i, (_, row) in enumerate(contracts_df.iterrows()):
        if i % 100 == 0:
            log.info(f"  Processing {i}/{total}…")
        feats = build_team_interest_features(
            posts_df, row["name"], row["Contract Year"]
        )
        feats["mlbam_id"]      = row["mlbam_id"]
        feats["contract_year"] = row["Contract Year"]
        rows.append(feats)

    return pd.DataFrame(rows)



In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Scrape MLBTR + extract team interest features")
    parser.add_argument("--years", nargs="*", type=int, default=list(range(2016, 2026)),
                        help="Contract years to scrape (space-separated)")
    parser.add_argument("--from-cache", action="store_true",
                        help="Skip HTTP requests — re-run NER on cached HTML only")
    args = parser.parse_args([]) # Pass an empty list to ignore IPython kernel arguments

    # If two years given, treat as range
    if len(args.years) == 2 and args.years[0] < args.years[1]:
        years = list(range(args.years[0], args.years[1] + 1))
    else:
        years = args.years

    log.info(f"Contract years: {years}")

    # ── 1. Load contracts (need player names + years) ─────────────────────────
    contracts = pd.read_csv("mlb_contracts.csv")
    contracts = contracts[contracts["mlbam_id"].notna()].copy()
    contracts["mlbam_id"] = contracts["mlbam_id"].astype(int)
    contracts = contracts[~contracts["name"].str.contains("Ohtani", na=False)]
    contracts = contracts[contracts["Contract Year"].isin(years)]
    log.info(f"Contracts to process: {len(contracts):,}")

    # ── 2. Scrape (or load from cache) ────────────────────────────────────────
    if not HAS_REQUESTS and not args.from_cache:
        log.error("requests not installed and --from-cache not set. Exiting.")
        return

    posts_df = collect_hot_stove_posts(years)

    # ── 3. Build team-interest features ───────────────────────────────────────
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    interest_df = build_all_team_interest(contracts, posts_df)

    interest_df.to_csv(OUTPUT_CSV, index=False)
    log.info(f"Team interest cache written → {OUTPUT_CSV}")
    print(interest_df.describe())


if __name__ == "__main__":
    main()


       n_teams_interested  n_rumor_mentions  has_big_market_interest  \
count         2246.000000       2246.000000              2246.000000   
mean             0.249332          1.007569                 0.051647   
std              1.269546          2.304529                 0.221363   
min              0.000000          0.000000                 0.000000   
25%              0.000000          0.000000                 0.000000   
50%              0.000000          0.000000                 0.000000   
75%              0.000000          1.000000                 0.000000   
max             18.000000         46.000000                 1.000000   

       first_rumor_doy  rumor_intensity       mlbam_id  contract_year  
count           2246.0      2246.000000    2246.000000    2246.000000  
mean             365.0         1.376670  567300.547195    2020.677204  
std                0.0        16.639949   90715.827281       2.851716  
min              365.0         0.000000  110073.000000    2016.